# 01 · ETL e validação de dados

**Objetivo.** Transformar os lotes brutos publicados do dataset *Default of Credit Card Clients* (UCI) em um conjunto processado, validado e versionado. Este notebook também abre o ciclo do pipeline (`cycle_id`) e fixa a `data_version` que os notebooks seguintes herdam.

**Entradas.**
- `config/config.yaml`
- `data/raw/source/default of credit card clients.xls`: arquivo original da UCI, baixado e conferido por SHA-256. Nunca é alterado.
- `data/raw/batches/lote_*.csv`: lotes publicados. Simulam a chegada de dados ao longo do tempo.

**Saídas.**
- `data/processed/dataset_processado.parquet`: dados limpos, com atributos derivados e a coluna `particao`.
- `data/processed/etl_metadata.json` e `reports/evidencias/<cycle_id>/01_etl.json`: relatório de validação.
- `reports/ciclo_atual.json`: identidade do ciclo (`cycle_id`, `data_version`), lida pelos notebooks 02 a 05.

**Metodologia.** Ingestão → versionamento → validação estrutural → nulos → duplicatas → valores inválidos → outliers → limpeza → engenharia de atributos → partição treino/validação/teste → gravação.

Toda a lógica reutilizável está em `src/ingestao.py` e `src/preprocessamento.py`. Este notebook orquestra e documenta.

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "config" / "config.yaml").is_file())
sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import display

from src import ingestao, preprocessamento as pp
from src.utils import load_config, start_cycle

pd.set_option("display.max_columns", 50)
cfg = load_config(ROOT)
TARGET = cfg["data"]["target"]

## 1. Ingestão e versionamento

`ensure_initial_batch` baixa a fonte se necessário, confere o checksum e garante que ao menos o `lote_001` foi publicado. A operação é idempotente.

A `data_version` é um hash SHA-256 sobre o nome e o conteúdo de cada lote publicado. Os mesmos lotes sempre geram a mesma versão. Um lote novo (`python -m src.ingestao --novo-lote`) gera uma versão nova.

In [2]:
batch_files = ingestao.ensure_initial_batch(cfg, ROOT)
data_version = ingestao.compute_data_version(batch_files)
ctx = start_cycle(cfg, data_version, [p.name for p in batch_files], ROOT)

print("cycle_id     :", ctx.cycle_id)
print("data_version :", ctx.data_version)
print("lotes        :", ctx.batches)

cycle_id     : ciclo_20261116T221924Z_a2d3d2d2
data_version : dv_a2d3d2d25ce5c5c8
lotes        : ['lote_001.csv']


## 2. Carga dos dados brutos

Na planilha original a primeira linha traz os rótulos `X1..X23, Y`, e o cabeçalho real está na segunda linha. Os lotes em CSV preservam os nomes originais. Aqui o alvo `default payment next month` é renomeado para o nome definido em `data.target`.

In [3]:
raw = pp.rename_target(ingestao.load_released_batches(cfg, ROOT), cfg)
print(raw.shape)
display(raw.head())
display(raw["lote"].value_counts().rename("linhas_por_lote"))

(18000, 26)


,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default.payment.next.month,lote
0,3,90000,2,2,2,34,0,0,0,0,0,0,29239,14027,13559,14331,14948,15549,1518,1500,1000,1000,1000,5000,0,lote_001
1,5,50000,1,2,1,57,-1,0,-1,0,0,0,8617,5670,35835,20940,19146,19131,2000,36681,10000,9000,689,679,0,lote_001
2,6,50000,1,1,2,37,0,0,0,0,0,0,64400,57069,57608,19394,19619,20024,2500,1815,657,1000,1000,800,0,lote_001
3,7,500000,1,1,2,29,0,0,0,0,0,0,367965,412023,445007,542653,483003,473944,55000,40000,38000,20239,13750,13770,0,lote_001
4,8,100000,2,2,2,23,0,-1,-1,0,0,-1,11876,380,601,221,-159,567,380,601,0,581,1687,1542,0,lote_001


lote
lote_001    18000
Name: linhas_por_lote, dtype: int64

### Dicionário das variáveis (fonte: UCI, Yeh & Lien, 2009)

| Variável | Significado |
|---|---|
| `LIMIT_BAL` | Limite de crédito concedido, em NT$ (dólar taiwanês), incluindo crédito do titular e da família |
| `SEX` | 1 = masculino, 2 = feminino |
| `EDUCATION` | 1 = pós-graduação, 2 = universidade, 3 = ensino médio, 4 = outros |
| `MARRIAGE` | 1 = casado, 2 = solteiro, 3 = outros |
| `AGE` | Idade em anos |
| `PAY_0, PAY_2..PAY_6` | Status de pagamento de set/2005 (`PAY_0`) até abr/2005 (`PAY_6`). -1 = pago em dia; 1..9 = meses de atraso. A nomenclatura original pula `PAY_1` |
| `BILL_AMT1..6` | Valor da fatura de set/2005 a abr/2005 (NT$) |
| `PAY_AMT1..6` | Valor pago de set/2005 a abr/2005 (NT$) |
| `default.payment.next.month` | **Alvo.** 1 = inadimplente no mês seguinte |

## 3. Validação estrutural, nulos, duplicatas e valores inválidos

`pp.validate` confere cada coluna contra a tabela `SCHEMA`, que define o domínio válido. Erros estruturais (coluna ausente, ID duplicado, alvo nulo ou fora de {0, 1}) interrompem o pipeline. Isso impede que um ciclo com dados corrompidos avance até a promoção de modelo.

In [4]:
report = pp.validate(raw, cfg)
if not report.is_valid:
    raise pp.DataValidationError(report.errors)

display(pd.Series({
    "linhas": report.n_rows,
    "colunas": report.n_columns,
    "colunas ausentes": report.missing_columns,
    "colunas inesperadas": report.unexpected_columns,
    "valores nulos": report.missing_values or "nenhum",
    "IDs duplicados": report.duplicate_ids,
    "linhas duplicadas (ignorando ID)": report.duplicate_rows_excluding_id,
    "valores fora do domínio documentado": report.invalid_values,
    "códigos não documentados": report.undocumented_codes,
    "balanço do alvo": report.class_balance,
}, name="valor").to_frame())

,valor
linhas,18000
colunas,26
colunas ausentes,[]
colunas inesperadas,[]
valores nulos,nenhum
IDs duplicados,0
linhas duplicadas (ignorando ID),15
valores fora do domínio documentado,"{'EDUCATION': 209, 'MARRIAGE': 34}"
códigos não documentados,"{'EDUCATION': {5: 173, 6: 28, 0: 8}, 'MARRIAGE..."
balanço do alvo,"{0: 0.7824, 1: 0.2176}"


**Leitura dos resultados.**

- **Nulos.** A fonte não tem valores ausentes. O pré-processador ainda inclui `SimpleImputer` para lotes futuros. Ele é ajustado só no treino.
- **Duplicatas.** Não há IDs repetidos. Existem linhas idênticas em todos os atributos e no alvo, mas com IDs diferentes. São clientes distintos com o mesmo perfil, em geral contas sem movimento. Por isso são **mantidas**.
- **Códigos não documentados.** `EDUCATION` ∈ {0, 5, 6} e `MARRIAGE` = 0 não constam no dicionário da UCI. São recodificados para a categoria "outros" (4 e 3). `PAY_x` = -2 e 0 também não constam no dicionário, mas são frequentes e interpretados na literatura como "sem consumo" e "crédito rotativo". Por isso são aceitos.
- **Desbalanceamento.** Cerca de 22% de inadimplentes. A acurácia seria enganosa, porque prever sempre "adimplente" já acerta cerca de 78%. Daí a métrica primária F1.

## 4. Outliers

Critério IQR: valores fora de [Q1 − 1,5·IQR, Q3 + 1,5·IQR]. É apenas diagnóstico.

In [5]:
outliers = pp.outlier_summary(raw)
display(outliers.sort_values("pct_outliers", ascending=False).reset_index(drop=True))

,coluna,q1,q3,limite_inf,limite_sup,min,max,n_outliers,pct_outliers
0,PAY_AMT4,291.75,4008.00,-5282.625,9582.375,0,528897,1773,0.0985
1,PAY_AMT6,100.00,4000.00,-5750.000,9850.000,0,528666,1756,0.0976
2,PAY_AMT5,203.00,4000.00,-5492.500,9695.500,0,417990,1744,0.0969
3,BILL_AMT5,1715.00,50249.00,-71086.000,123050.000,-81334,927171,1618,0.0899
4,PAY_AMT2,827.00,5000.00,-5432.500,11259.500,0,1684259,1616,0.0898
5,BILL_AMT4,2256.50,54154.75,-75590.875,132002.125,-170000,891586,1595,0.0886
6,PAY_AMT1,941.00,5000.00,-5147.500,11088.500,0,493358,1595,0.0886
7,BILL_AMT6,1206.75,49355.50,-71016.375,121578.625,-339603,961664,1585,0.0881
8,PAY_AMT3,390.00,4560.00,-5865.000,10815.000,0,896040,1562,0.0868
9,BILL_AMT3,2574.75,59565.00,-82910.625,145050.375,-61506,1664089,1471,0.0817


**Decisão.** Os outliers **não são removidos**. Faturas e pagamentos altos são clientes reais de alto limite, e não erros de digitação: todos os valores respeitam o domínio validado na seção 3. Removê-los enviesaria o modelo contra exatamente esse público. A Random Forest é insensível à escala. Para a Regressão Logística, a padronização é ajustada só no treino.

## 5. Limpeza

Aplica a recodificação dos códigos não documentados e remove linhas que continuem inválidas. Se a fração removida passar de `data.validation.max_invalid_row_fraction`, o ETL falha.

In [6]:
clean, cleaning_log = pp.clean(raw, cfg)
display(pd.Series(cleaning_log, name="contagem").to_frame())
display(clean["EDUCATION"].value_counts().sort_index().rename("EDUCATION após limpeza"))

,contagem
recodificados_EDUCATION,209
recodificados_MARRIAGE,34
linhas_removidas,0


EDUCATION
1    6368
2    8388
3    2963
4     281
Name: EDUCATION após limpeza, dtype: int64

## 6. Engenharia de atributos

Cada atributo usa apenas valores da própria linha. Não há estatística agregada do dataset, logo não há vazamento entre treino e teste.

| Atributo | Definição | Justificativa |
|---|---|---|
| `UTILIZACAO_ATUAL` | `BILL_AMT1 / LIMIT_BAL` | Uso do limite no último mês |
| `UTILIZACAO_MEDIA` | média(`BILL_AMT1..6`) / `LIMIT_BAL` | Uso típico do limite no semestre |
| `RAZAO_PAGAMENTO` | Σ`PAY_AMT` / Σ max(`BILL_AMT`, 0), limitado a [0, 5] | Quanto da dívida o cliente costuma quitar |
| `MESES_EM_ATRASO` | nº de meses com `PAY_x` ≥ 1 | Frequência de atraso |
| `ATRASO_MAXIMO` | max(`PAY_x`) | Gravidade do pior atraso |
| `TENDENCIA_FATURA` | (`BILL_AMT1` − `BILL_AMT6`) / `LIMIT_BAL` | Endividamento crescente ou decrescente |

In [7]:
features_df = pp.engineer_features(clean)
display(features_df[pp.ENGINEERED].describe().T)
display(features_df.groupby(TARGET)[pp.ENGINEERED].mean().T.rename(columns={0: "adimplente", 1: "inadimplente"}))

,count,mean,std,min,25%,50%,75%,max
UTILIZACAO_ATUAL,18000.0,0.422588,0.411316,-0.619892,0.021167,0.308717,0.829788,5.3095
UTILIZACAO_MEDIA,18000.0,0.371397,0.350419,-0.200154,0.029391,0.278926,0.687454,2.6738
RAZAO_PAGAMENTO,18000.0,0.408374,0.607141,0.000000,0.043202,0.107718,0.714502,5.0000
MESES_EM_ATRASO,18000.0,0.837111,1.558397,0.000000,0.000000,0.000000,1.000000,6.0000
ATRASO_MAXIMO,18000.0,0.437222,1.353737,-2.000000,0.000000,0.000000,2.000000,8.0000
TENDENCIA_FATURA,18000.0,0.105939,0.303836,-1.614200,-0.028645,0.006950,0.182192,5.3095


default.payment.next.month,adimplente,inadimplente
UTILIZACAO_ATUAL,0.404056,0.489240
UTILIZACAO_MEDIA,0.350618,0.446129
RAZAO_PAGAMENTO,0.430761,0.327860
MESES_EM_ATRASO,0.514627,1.996936
ATRASO_MAXIMO,0.205978,1.268897
TENDENCIA_FATURA,0.108831,0.095540


## 7. Protocolo de avaliação: treino / validação / teste

O protocolo é definido **antes** de qualquer treino.

- **Partição por hash do ID.** Cada cliente recebe um número em [0, 1] derivado de `sha256(random_state:ID)`. Abaixo de `test_size` vai para **teste**. Entre `test_size` e `test_size + validation_size` vai para **validação**. O restante vai para **treino**. Com a configuração atual a divisão é 60/20/20.
- **Por que hash e não `train_test_split`?** Quando um lote novo chega, um sorteio aleatório reembaralharia tudo, e clientes do teste do ciclo 1 poderiam virar treino no ciclo 2. Com o hash, cada cliente fica sempre na mesma partição. O teste continua isolado entre ciclos.
- **Uso de cada partição.** O treino ajusta o pré-processador, os seletores de atributos e os modelos. A validação decide a competição entre modelos e a promoção. O teste é usado **só** na avaliação final, nunca para decisões recorrentes.
- O pré-processamento com parâmetros (imputação, one-hot, padronização) fica em `pp.build_preprocessor` e é ajustado apenas no treino.

In [8]:
dataset = features_df.assign(particao=pp.assign_split(features_df[cfg["data"]["id_column"]], cfg))

split_table = dataset.groupby("particao").agg(linhas=(TARGET, "size"), taxa_inadimplencia=(TARGET, "mean"))
split_table["fracao"] = split_table["linhas"] / len(dataset)
display(split_table.round(4))

assert pp.assign_split(dataset[cfg["data"]["id_column"]], cfg).equals(dataset["particao"]), "particao deterministica"
assert not set(dataset.loc[dataset.particao == "teste", "ID"]) & set(dataset.loc[dataset.particao != "teste", "ID"]), "teste isolado"

,linhas,taxa_inadimplencia,fracao
particao,,,
teste,3536,0.2067,0.1964
treino,10851,0.2154,0.6028
validacao,3613,0.2347,0.2007


## 8. Separação X / y e verificação do pré-processador

Separa atributos e alvo e ajusta o `ColumnTransformer` **somente no treino**. Validação e teste apenas recebem `transform`. Esta célula comprova que o pré-processador funciona. O ajuste definitivo acontece dentro do pipeline de cada modelo, no notebook 03.

In [9]:
features = pp.feature_columns(cfg, dataset)
train = dataset[dataset.particao == "treino"]
X_train, y_train = train[features], train[TARGET]

preprocessor = pp.build_preprocessor(features).fit(X_train)
X_valid = preprocessor.transform(dataset.loc[dataset.particao == "validacao", features])

print("atributos candidatos :", len(features))
print("após one-hot         :", len(preprocessor.get_feature_names_out()))
print("X_train              :", X_train.shape, "| X_valid transformado:", X_valid.shape)

atributos candidatos : 29
após one-hot         : 35
X_train              : (10851, 29) | X_valid transformado: (3613, 35)


## 9. Gravação do dataset processado

In [10]:
path = pp.save_processed(dataset, ctx, cfg, {
    "validacao": report.to_dict(),
    "limpeza": cleaning_log,
    "outliers": outliers.set_index("coluna")["n_outliers"].to_dict(),
    "atributos_derivados": pp.ENGINEERED,
    "atributos_candidatos": features,
}, ROOT)
print(path.relative_to(ROOT))

2026-11-16 19:19:25,222 INFO src.preprocessamento: Dataset processado salvo: dataset_processado.parquet (18000 linhas, dv_a2d3d2d25ce5c5c8)


data/processed/dataset_processado.parquet


## Conclusões

- O ciclo foi aberto com um `cycle_id` e uma `data_version` derivada do conteúdo dos lotes. Os dois seguem para os notebooks 02 a 05 via `reports/ciclo_atual.json`.
- A validação bloqueia o pipeline em caso de erro estrutural. Códigos não documentados são recodificados de forma explícita e registrada.
- Outliers são reais e foram mantidos. Duplicatas por perfil são clientes distintos e também foram mantidas.
- A partição por hash garante que o conjunto de teste continue isolado quando novos lotes chegam.
- Nenhuma transformação com parâmetros foi ajustada fora do treino.

**Próximo passo:** `02_votacao_features.ipynb`, com três métodos de seleção de atributos e votação.